In [73]:
!pip install gradio chromadb openai python-dotenv pymupdf python-docx beautifulsoup4 lxml tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 13.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [python-docx] [python-docx]


In [74]:
# ──────────────────────────────────────────────
# STEP 1 – Imports
# ──────────────────────────────────────────────

import os
import re
import json
import uuid
from pathlib import Path

import fitz
import tiktoken
import chromadb
import gradio as gr

from bs4 import BeautifulSoup
from docx import Document
from openai import OpenAI
from dotenv import load_dotenv

In [44]:
# ──────────────────────────────────────────────
# STEP 2A – OpenAI Client (Local)
# ──────────────────────────────────────────────

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized.")

OpenAI client initialized.


In [ ]:
# ──────────────────────────────────────────────
# STEP 2B – OpenAI Client (Colab)
# ──────────────────────────────────────────────

os.environ["OPENAI_API_KEY"] = "your_key_here"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [45]:
# ──────────────────────────────────────────────
# STEP 3 – Load Chroma Vector Store
# ──────────────────────────────────────────────

BASE_DIR = Path(".")
M2_OUTPUT_DIR = BASE_DIR / "milestone2_outputs"
CHROMA_DIR = M2_OUTPUT_DIR / "chroma_db"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection("legal_ai_chunks")

print("Loaded collection.")
print("Collection count:", collection.count())

Loaded collection.
Collection count: 308


In [75]:
# ──────────────────────────────────────────────
# STEP 4 – Token Counter
# ──────────────────────────────────────────────

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(text))

In [49]:
# ──────────────────────────────────────────────
# STEP 5 – Parsing Helpers
# ──────────────────────────────────────────────

def clean_extracted_text(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def remove_repeated_lines(text, min_repeats=3):
    lines = [line.strip() for line in text.splitlines()]
    counts = {}
    for line in lines:
        if line:
            counts[line] = counts.get(line, 0) + 1

    repeated = {line for line, c in counts.items() if c >= min_repeats}

    cleaned_lines = []
    for line in lines:
        if line in repeated and len(line) < 120:
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [50]:
# ──────────────────────────────────────────────
# STEP 6 – File Parsers
# ──────────────────────────────────────────────

def parse_pdf(file_path):
    doc = fitz.open(str(file_path))
    pages = []

    for page in doc:
        page_text = page.get_text("text")
        pages.append(page_text)

    full_text = "\n\n".join(pages)
    full_text = clean_extracted_text(full_text)
    full_text = remove_repeated_lines(full_text)
    return full_text

def parse_docx(file_path):
    doc = Document(str(file_path))
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    full_text = "\n\n".join(paragraphs)
    return clean_extracted_text(full_text)

def parse_html(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    return clean_extracted_text(text)

def parse_uploaded_document(file_path):
    suffix = Path(file_path).suffix.lower()

    if suffix == ".pdf":
        return parse_pdf(file_path)
    elif suffix == ".docx":
        return parse_docx(file_path)
    elif suffix in {".html", ".htm"}:
        return parse_html(file_path)
    elif suffix == ".doc":
        raise ValueError("Legacy .doc files should be converted to .pdf or .docx first.")
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

In [53]:
# ──────────────────────────────────────────────
# STEP 7 – Clause-Aware Chunking
# ──────────────────────────────────────────────

CLAUSE_HEADER_PATTERN = re.compile(
    r'(?im)^(section\s+\d+(\.\d+)*|article\s+[ivx\d]+|'
    r'\d+(\.\d+)*\s+[A-Z][^\n]*|[A-Z][A-Z \-/]{4,})$'
)

def split_into_sections(text):
    lines = text.splitlines()
    sections = []
    current = []

    for line in lines:
        stripped = line.strip()
        if CLAUSE_HEADER_PATTERN.match(stripped):
            if current:
                sections.append("\n".join(current).strip())
                current = []
        current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return [s for s in sections if s.strip()]

def build_chunk_from_words(words, chunk_index):
    chunk_text = " ".join(words).strip()
    return {
        "chunk_index": chunk_index,
        "chunk_text": chunk_text,
        "token_count": count_tokens(chunk_text)
    }

def get_overlap_words(words, target_overlap_tokens=175):
    overlap_words = []
    for word in reversed(words):
        overlap_words.insert(0, word)
        if count_tokens(" ".join(overlap_words)) >= target_overlap_tokens:
            break
    return overlap_words

def chunk_document(text, chunk_size=900, overlap=175):
    sections = split_into_sections(text)

    chunks = []
    current_words = []
    chunk_index = 0

    for section in sections:
        section_words = section.split()
        trial_words = current_words + section_words
        trial_text = " ".join(trial_words)

        if count_tokens(trial_text) <= chunk_size:
            current_words = trial_words
        else:
            if current_words:
                chunks.append(build_chunk_from_words(current_words, chunk_index))
                chunk_index += 1
                overlap_words = get_overlap_words(current_words, overlap)
                current_words = overlap_words + section_words
            else:
                # section itself too big, split roughly by token count
                temp_words = section_words[:]
                while temp_words:
                    candidate = []
                    while temp_words:
                        trial = candidate + [temp_words[0]]
                        if count_tokens(" ".join(trial)) <= chunk_size:
                            candidate.append(temp_words.pop(0))
                        else:
                            break

                    if not candidate and temp_words:
                        candidate = [temp_words.pop(0)]

                    chunks.append(build_chunk_from_words(candidate, chunk_index))
                    chunk_index += 1

                    overlap_words = get_overlap_words(candidate, overlap)
                    temp_words = overlap_words + temp_words if temp_words else temp_words

                    if len(temp_words) == len(overlap_words):
                        break

                current_words = []

    if current_words:
        chunks.append(build_chunk_from_words(current_words, chunk_index))

    return chunks

In [55]:
# ──────────────────────────────────────────────
# STEP 4 – Embedding Function
# ──────────────────────────────────────────────

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [54]:
# ──────────────────────────────────────────────
# STEP 8 – Clause Tagging
# ──────────────────────────────────────────────

CLAUSE_PATTERNS = {
    "Payment / financial terms": [
        "payment", "fees", "rent", "deposit", "compensation", "salary", "invoice", "charges"
    ],
    "Termination / cancellation": [
        "terminate", "termination", "cancel", "expiration", "renewal", "breach"
    ],
    "Liability / indemnification": [
        "liability", "indemnify", "indemnification", "damages", "losses"
    ],
    "Confidentiality / non-disclosure": [
        "confidential", "confidentiality", "non-disclosure", "nda", "proprietary information"
    ],
    "IP ownership / licensing": [
        "intellectual property", "ip ownership", "license", "licensing", "copyright", "patent"
    ],
    "Governing law / jurisdiction": [
        "governing law", "jurisdiction", "venue", "state of", "laws of"
    ],
    "Dispute resolution / arbitration": [
        "dispute", "arbitration", "arbitrate", "litigation", "court", "jury trial"
    ],
    "Warranties / representations": [
        "warranty", "warranties", "representations", "as is", "disclaimer"
    ]
}

def tag_chunk_text(text):
    lowered = text.lower()
    tags = []

    for tag, patterns in CLAUSE_PATTERNS.items():
        if any(p in lowered for p in patterns):
            tags.append(tag)

    return tags

In [56]:
# ──────────────────────────────────────────────
# STEP 10 – Build Vector Store for Uploaded File
# ──────────────────────────────────────────────

def build_temp_collection_from_upload(file_path):
    file_path = Path(file_path)
    source_document = file_path.name

    raw_text = parse_uploaded_document(file_path)
    chunks = chunk_document(raw_text)

    if not chunks:
        raise ValueError("No chunks were created from the uploaded document.")

    collection_name = f"uploaded_doc_{uuid.uuid4().hex[:8]}"

    collection = chroma_client.create_collection(name=collection_name)

    ids = []
    documents = []
    metadatas = []
    embeddings = []

    for chunk in chunks:
        chunk_id = f"{source_document}::chunk_{chunk['chunk_index']}"
        clause_tags = tag_chunk_text(chunk["chunk_text"])

        ids.append(chunk_id)
        documents.append(chunk["chunk_text"])
        metadatas.append({
            "source_document": source_document,
            "chunk_index": int(chunk["chunk_index"]),
            "token_count": int(chunk["token_count"]),
            "clause_tags_str": " | ".join(clause_tags) if clause_tags else ""
        })
        embeddings.append(get_embedding(chunk["chunk_text"]))

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )

    return {
        "collection_name": collection_name,
        "source_document": source_document,
        "raw_text": raw_text,
        "num_chunks": len(chunks)
    }

In [57]:
# ──────────────────────────────────────────────
# STEP 11 – Retrieval for Uploaded File
# ──────────────────────────────────────────────

def retrieve_top_k_uploaded(query, collection_name, top_k=5, initial_k=15):
    collection = chroma_client.get_collection(collection_name)
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=initial_k
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []

    rows = []
    for i in range(len(docs)):
        rows.append({
            "id": ids[i],
            "document": docs[i],
            "metadata": metas[i],
            "distance": distances[i] if distances else None
        })

    return rows[:top_k]

In [58]:
# ──────────────────────────────────────────────
# STEP 12 – Context Builder
# ──────────────────────────────────────────────

def build_context(rows):
    if not rows:
        return "NO_RELEVANT_CONTEXT_FOUND"

    parts = []
    for i, row in enumerate(rows, start=1):
        meta = row["metadata"]
        parts.append(
            f"[Chunk {i}]\n"
            f"Chunk ID: {row['id']}\n"
            f"Source Document: {meta.get('source_document')}\n"
            f"Chunk Index: {meta.get('chunk_index')}\n"
            f"Clause Tags: {meta.get('clause_tags_str')}\n"
            f"Distance: {row.get('distance')}\n"
            f"Text:\n{row['document']}\n"
        )

    return "\n\n".join(parts)

In [59]:
# ──────────────────────────────────────────────
# STEP 13 – System Prompts
# ──────────────────────────────────────────────

QA_SYSTEM_PROMPT = """
You are a legal document assistant.

Rules:
- Use ONLY the retrieved contract context.
- Do NOT provide legal advice.
- Do NOT invent facts, clauses, or interpretations.
- If the context does not support an answer, say so clearly.
- Keep the answer understandable to a non-lawyer.

Required format:

Plain-English Answer:
<answer>

Key Risks / Important Notes:
- <bullet>
- <bullet>

Supporting Citations:
- [Chunk X] <why this chunk supports the answer>
- [Chunk Y] <why this chunk supports the answer>

Disclaimer:
This is not legal advice and does not replace review by an attorney.
"""

SUMMARY_SYSTEM_PROMPT = """
You are a legal document assistant.

Rules:
- Summarize ONLY the provided contract context.
- Do NOT provide legal advice.
- Do NOT invent facts or obligations not supported by the text.
- Keep the summary understandable to a non-lawyer.

Required format:

Plain-English Summary:
<short summary of what the document is about>

Key Obligations / Important Terms:
- <bullet>
- <bullet>
- <bullet>

Potential Risks / Things to Watch:
- <bullet>
- <bullet>
- <bullet>

Supporting Citations:
- [Chunk X] <why this chunk matters>
- [Chunk Y] <why this chunk matters>

Disclaimer:
This is not legal advice and does not replace review by an attorney.
"""

CLAUSE_SYSTEM_PROMPT = """
You are a legal document assistant.

Rules:
- Explain ONLY the retrieved clause text.
- Do NOT provide legal advice.
- Do NOT invent terms that are not present in the text.
- Keep the explanation understandable to a non-lawyer.

Required format:

Clause Summary:
<plain-English explanation>

Why It Matters:
- <bullet>
- <bullet>

Supporting Citations:
- [Chunk X] <why this chunk supports the explanation>
- [Chunk Y] <why this chunk supports the explanation>

Disclaimer:
This is not legal advice and does not replace review by an attorney.
"""

In [60]:
# ──────────────────────────────────────────────
# STEP 14 – Chat Helper
# ──────────────────────────────────────────────

CHAT_MODEL = "gpt-4.1-mini"

def generate_grounded_response(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

In [61]:
# ──────────────────────────────────────────────
# STEP 15 – Question Answering
# ──────────────────────────────────────────────

def answer_question_uploaded(question, collection_name, source_document):
    if not collection_name:
        return "Please upload a document first."

    if not question or question.strip() == "":
        return "Please enter a question."

    rows = retrieve_top_k_uploaded(question, collection_name, top_k=5, initial_k=15)
    context = build_context(rows)

    if context == "NO_RELEVANT_CONTEXT_FOUND":
        return (
            "Plain-English Answer:\n"
            "I could not find relevant supporting text in the uploaded document.\n\n"
            "Key Risks / Important Notes:\n"
            "- No reliable answer can be given from the retrieved evidence.\n\n"
            "Supporting Citations:\n"
            "- None.\n\n"
            "Disclaimer:\n"
            "This is not legal advice and does not replace review by an attorney."
        )

    user_prompt = f"""
Uploaded Document:
{source_document}

User Question:
{question}

Retrieved Contract Context:
{context}
"""

    return generate_grounded_response(QA_SYSTEM_PROMPT, user_prompt)

In [62]:
# ──────────────────────────────────────────────
# STEP 16 – Document Summarization (RAG)
# ──────────────────────────────────────────────

def summarize_document_uploaded(collection_name, source_document):
    if not collection_name:
        return "Please upload a document first."

    summary_queries = [
        "What is this contract about?",
        "What are the main obligations in this agreement?",
        "What are the payment terms?",
        "How can the agreement be terminated?",
        "What are the major risks or limitations?"
    ]

    all_rows = []
    seen_ids = set()

    for query in summary_queries:
        rows = retrieve_top_k_uploaded(query, collection_name, top_k=3, initial_k=10)
        for row in rows:
            if row["id"] not in seen_ids:
                all_rows.append(row)
                seen_ids.add(row["id"])

    context = build_context(all_rows)

    if context == "NO_RELEVANT_CONTEXT_FOUND":
        return (
            "Plain-English Summary:\n"
            "I could not find enough relevant supporting text to summarize the uploaded document.\n\n"
            "Key Obligations / Important Terms:\n"
            "- None identified from retrieved context.\n\n"
            "Potential Risks / Things to Watch:\n"
            "- Summary unavailable due to lack of retrieved evidence.\n\n"
            "Supporting Citations:\n"
            "- None.\n\n"
            "Disclaimer:\n"
            "This is not legal advice and does not replace review by an attorney."
        )

    user_prompt = f"""
Uploaded Document:
{source_document}

Task:
Create a plain-English summary of this uploaded legal document using only the provided context.

Contract Context:
{context}
"""

    return generate_grounded_response(SUMMARY_SYSTEM_PROMPT, user_prompt)

In [63]:
# ──────────────────────────────────────────────
# STEP 17 – Automatic Key Clauses
# ──────────────────────────────────────────────

def show_key_clauses_uploaded(collection_name, source_document):
    if not collection_name:
        return "Please upload a document first."

    clause_queries = {
        "Termination": "termination of agreement",
        "Payment": "payment terms",
        "Liability": "liability or indemnification",
        "Confidentiality": "confidentiality obligations",
        "IP Ownership": "intellectual property ownership"
    }

    outputs = []

    for clause_name, query in clause_queries.items():
        rows = retrieve_top_k_uploaded(query, collection_name, top_k=3, initial_k=10)
        context = build_context(rows)

        if context == "NO_RELEVANT_CONTEXT_FOUND":
            continue

        user_prompt = f"""
Uploaded Document:
{source_document}

Clause Category:
{clause_name}

Task:
Explain this clause category in plain English using only the provided contract context.

Contract Context:
{context}
"""

        explanation = generate_grounded_response(CLAUSE_SYSTEM_PROMPT, user_prompt)
        outputs.append(f"=== {clause_name} ===\n{explanation}")

    if not outputs:
        return (
            "Clause Summary:\n"
            "I could not identify key clauses from the uploaded document.\n\n"
            "Why It Matters:\n"
            "- No reliable clause explanation can be given from the retrieved evidence.\n\n"
            "Supporting Citations:\n"
            "- None.\n\n"
            "Disclaimer:\n"
            "This is not legal advice and does not replace review by an attorney."
        )

    return "\n\n".join(outputs)

In [64]:
# ──────────────────────────────────────────────
# STEP 18 – Upload Handler
# ──────────────────────────────────────────────

def handle_upload(file):
    if file is None:
        return "No file uploaded.", None

    try:
        doc_info = build_temp_collection_from_upload(file.name)
        status = (
            f"Uploaded and indexed: {doc_info['source_document']}\n"
            f"Chunks created: {doc_info['num_chunks']}\n"
            f"Temporary collection: {doc_info['collection_name']}"
        )
        return status, doc_info

    except Exception as e:
        return f"Upload failed: {str(e)}", None

In [65]:
# ──────────────────────────────────────────────
# STEP 19 – Action Handlers
# ──────────────────────────────────────────────

def handle_summary(doc_info):
    if not doc_info:
        return "Please upload a document first."

    return summarize_document_uploaded(
        collection_name=doc_info["collection_name"],
        source_document=doc_info["source_document"]
    )

def handle_question(question, doc_info):
    if not doc_info:
        return "Please upload a document first."

    return answer_question_uploaded(
        question=question,
        collection_name=doc_info["collection_name"],
        source_document=doc_info["source_document"]
    )

def handle_key_clauses(doc_info):
    if not doc_info:
        return "Please upload a document first."

    return show_key_clauses_uploaded(
        collection_name=doc_info["collection_name"],
        source_document=doc_info["source_document"]
    )

In [77]:
# ──────────────────────────────────────────────
# STEP 20 – Gradio App
# ──────────────────────────────────────────────

with gr.Blocks(title="Legal AI MVP") as demo:
    gr.Markdown("# Legal AI MVP")
    gr.Markdown(
        "Upload a legal document to receive a plain-English summary, "
        "ask questions about it, and automatically surface key clauses."
    )

    doc_state = gr.State()

    with gr.Row():
        file_input = gr.File(label="Upload Contract")
        upload_button = gr.Button("Process Document")

    upload_status = gr.Textbox(label="Upload Status", lines=4)

    question_box = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: Can either party terminate this agreement?",
        lines=3
    )

    with gr.Row():
        summarize_button = gr.Button("Summarize Document")
        ask_button = gr.Button("Ask Question")
        clauses_button = gr.Button("Show Key Clauses")

    output_box = gr.Textbox(label="Output", lines=28)

    upload_button.click(
        fn=handle_upload,
        inputs=[file_input],
        outputs=[upload_status, doc_state]
    )

    summarize_button.click(
        fn=handle_summary,
        inputs=[doc_state],
        outputs=[output_box]
    )

    ask_button.click(
        fn=handle_question,
        inputs=[question_box, doc_state],
        outputs=[output_box]
    )

    clauses_button.click(
        fn=handle_key_clauses,
        inputs=[doc_state],
        outputs=[output_box]
    )

In [78]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://a8564c5c070421956e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [40]:
# ──────────────────────────────────────────────
# STEP 18 – Debug App
# ──────────────────────────────────────────────

def run_legal_ai_debug(selected_document, mode, question, clause_type):
    output = run_legal_ai(selected_document, mode, question, clause_type)
    context = get_debug_context(selected_document, mode, question, clause_type)
    return output, context

In [41]:
demo_debug = gr.Interface(
    fn=run_legal_ai_debug,
    inputs=[
        gr.Dropdown(
            choices=available_documents,
            label="Select Document"
        ),
        gr.Radio(
            choices=["Ask a Question", "Summarize Document", "Highlight Clause"],
            value="Ask a Question",
            label="Mode"
        ),
        gr.Textbox(label="Question", lines=3),
        gr.Dropdown(
            choices=CLAUSE_OPTIONS,
            label="Clause Type"
        )
    ],
    outputs=[
        gr.Textbox(label="Output", lines=24),
        gr.Textbox(label="Retrieved Context", lines=28)
    ],
    title="Legal AI MVP (Debug View)",
    description="Shows both the user-facing output and the retrieved supporting chunks."
)

In [42]:
demo_debug.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# ──────────────────────────────────────────────
# STEP 21 – Save Example Outputs
# ──────────────────────────────────────────────

def save_example_outputs(doc_info):
    if not doc_info:
        return "Please upload a document first."

    examples = {
        "summary": summarize_document_uploaded(doc_info["collection_name"], doc_info["source_document"]),
        "question_example": answer_question_uploaded(
            "How can either party terminate this agreement?",
            doc_info["collection_name"],
            doc_info["source_document"]
        ),
        "key_clauses": show_key_clauses_uploaded(doc_info["collection_name"], doc_info["source_document"])
    }

    output_path = M3_OUTPUT_DIR / "example_outputs.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(examples, f, indent=2)

    return f"Saved example outputs to {output_path}"

In [ ]:
# ──────────────────────────────────────────────
# STEP 18 – Milestone 3 Summary
# ──────────────────────────────────────────────

print("MILESTONE 3 SUMMARY")
print("=" * 50)
print("Modes implemented:")
print("- Ask a Question")
print("- Summarize Document")
print("- Highlight Clause")
print("\nCapabilities:")
print("- Uses Chroma vector store")
print("- Uses embeddings for retrieval")
print("- Uses OpenAI chat model for grounded generation")
print("- Returns plain-English outputs")
print("- Includes citations")
print("- Includes legal disclaimer")
print("- Exposes functionality through Gradio UI")